In [4]:
import time
from concurrent.futures import ThreadPoolExecutor
def get_embedding(text:str) -> list:
    time.sleep(0.3)
    return [0.1] * 768
docs = [f'' for i in range(20)]
with ThreadPoolExecutor(max_workers=10) as pool:
    vectors = list(pool.map(get_embedding,docs))
print(len(vectors))


20


In [8]:
from concurrent.futures import ThreadPoolExecutor,as_completed
import random,time
def ask_model(model:str,prompt:str) -> dict:
    delay = random.uniform(1,4)
    time.sleep(delay)
    return {"model": model, "answer": f"{model}的回答", "latency": delay}
models = ["GPT-4o", "Claude-3.5", "Qwen-Max"]
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {pool.submit(ask_model,m,'解释量子计算'):m for m in models}
    for future in as_completed(futures):
        result = future.result()
        print(f" {result['model']} 率先返回 ({result['latency']:.1f}s)")
        break

 Claude-3.5 率先返回 (2.2s)


In [9]:
import threading,time
from concurrent.futures import ThreadPoolExecutor
semaphore = threading.Semaphore(3)
def safe_call(task_id:int):
    with semaphore:
        print(f"🟢 任务 {task_id} 开始")
        time.sleep(2)
        print(f"🔴 任务 {task_id} 结束")
with ThreadPoolExecutor(max_workers=20) as pool:
    list(pool.map(safe_call,range(15)))

🟢 任务 0 开始
🟢 任务 1 开始
🟢 任务 2 开始
🔴 任务 1 结束
🟢 任务 3 开始
🔴 任务 0 结束
🟢 任务 4 开始
🔴 任务 2 结束
🟢 任务 5 开始
🔴 任务 4 结束🔴 任务 3 结束
🟢 任务 6 开始

🟢 任务 7 开始
🔴 任务 5 结束
🟢 任务 8 开始
🔴 任务 7 结束🔴 任务 6 结束
🔴 任务 8 结束
🟢 任务 9 开始
🟢 任务 10 开始

🟢 任务 11 开始
🔴 任务 9 结束🔴 任务 10 结束
🔴 任务 11 结束
🟢 任务 12 开始
🟢 任务 13 开始

🟢 任务 14 开始
🔴 任务 13 结束
🔴 任务 14 结束
🔴 任务 12 结束


In [10]:
import threading
from concurrent.futures import ThreadPoolExecutor
total_cost = 0.0
lock = threading.Lock()
def record_cost(cost:float):
    global total_cost
    with lock:
        total_cost += cost
with ThreadPoolExecutor(max_workers=20) as pool:
    list(pool.map(record_cost,[0.01]*100))
print(f"总花费: ${total_cost:.4f}")

总花费: $1.0000


In [11]:
from concurrent.futures import ThreadPoolExecutor,as_completed
def risky_call(i:int):
    if i == 3:
        raise ValueError(f"任务 {i} API 返回 500")
    return f"任务 {i} 成功"
with ThreadPoolExecutor(max_workers=5) as pool:
    futures = [pool.submit(risky_call,i) for i in range(6)]
    for future in as_completed(futures):
        try:
            print(f"✅ {future.result()}")
        except Exception as e:
            print(f"❌ {e}")

❌ 任务 3 API 返回 500
✅ 任务 5 成功
✅ 任务 4 成功
✅ 任务 2 成功
✅ 任务 0 成功
✅ 任务 1 成功


In [13]:
import time
from concurrent.futures import ThreadPoolExecutor,TimeoutError,as_completed
def slow_api_call(task_id:int):
    if task_id == 2:
        time.sleep(10)
    time.sleep(1)
    return f"任务 {task_id} 成功"
with ThreadPoolExecutor(max_workers=5) as pool:
    futures = [pool.submit(slow_api_call,i) for i in range(5)]
    for future in as_completed(futures):
        try:
            result = future.result(timeout=3)
            print(f"✅ {result}")
        except TimeoutError:
            print(f" 任务超时！已放弃等待（线程会在后台自行结束）")
        except Exception as e:
            print(f"❌ 其他错误: {e}")


✅ 任务 1 成功
✅ 任务 0 成功
✅ 任务 4 成功
✅ 任务 3 成功
✅ 任务 2 成功


In [15]:
import time 
from concurrent.futures import ThreadPoolExecutor
def embed_batch(batch:list[str]) -> int:
    time.sleep(1)
    return len(batch)
texts = [f"文本 {i}" for i in range(50)]
BATCH_SIZE = 10
batches = [texts[i:i+BATCH_SIZE] for i in range(0,len(texts),BATCH_SIZE)]
print(f"共 {len(texts)} 条文本，分成了 {len(batches)} 批")
with ThreadPoolExecutor(max_workers=5) as pool:
    results = list(pool.map(embed_batch,batches))
print(f"✅ 全部完成，共生成 {sum(results)} 个向量")


共 50 条文本，分成了 5 批
✅ 全部完成，共生成 50 个向量
